# Failure Mode 9: Verification Skipped

> Before starting, read the [project README](../../README.md) for setup instructions and background on traces, scorers, and failure modes.

The agent completes an action but doesn't use available tools to verify the result. This is about checking your own work — an agent that books a flight but never confirms the booking actually went through is skipping a step that its toolset supports.

### Why a custom scorer?

No existing MLflow scorer checks whether an agent verified the results of its own actions. Built-in scorers evaluate tool correctness, goal achievement, or response quality — none assess whether the agent used available verification tools after performing a state-changing action.

Whether verification was warranted depends on context: did the action tool's response already confirm success, or did it return a minimal result that leaves the outcome uncertain? A deterministic check ("was `verify_booking` called?") would be too rigid — sometimes skipping verification is the right call. An LLM judge can reason about whether verification was actually needed.

This notebook uses `make_judge()` — the same pattern introduced in [Graceful Refusal](../05_graceful_refusal/05_graceful_refusal.ipynb) and used again in [Repeated Action Loop](../07_repeated_action_loop/07_repeated_action_loop.ipynb).

| Scorer | Source | Needs expectations? | What it checks |
|---|---|---|---|
| Custom `make_judge()` | Custom | No | Did the agent verify its action when verification was warranted? |

For a detailed explanation of this failure mode and how the custom judge works, see [verification_skipped.md](verification_skipped.md).

### Prerequisites and setup

Start a local MLflow server before running this notebook:

```bash
mlflow server --host 127.0.0.1 --port 5000
```

In [ ]:
import sys
from pathlib import Path

sys.path.insert(0, str(Path("../..").resolve()))

import mlflow
from mlflow.entities import SpanType
from tools import TRAVEL_AGENT_TOOLS
from utils import print_eval_results

mlflow.set_tracking_uri("http://localhost:5000")
mlflow.set_experiment("agentic-evaluation")
mlflow.tracing.disable_notebook_display()

EXPERIMENT = mlflow.get_experiment_by_name("agentic-evaluation")

# Clean up old traces for this failure mode
client = mlflow.MlflowClient()
old_traces = mlflow.search_traces(
    experiment_ids=[EXPERIMENT.experiment_id],
    filter_string="tags.failure_mode = 'verification_skipped'",
    return_type="list",
)
if old_traces:
    client.delete_traces(
        experiment_id=EXPERIMENT.experiment_id,
        trace_ids=[t.info.trace_id for t in old_traces],
    )
    print(f"Cleaned up {len(old_traces)} old traces.")

### Create traces

We create synthetic traces for a travel booking agent that has `verify_booking` in its toolset. Three scenarios:

- **Unverified booking (fail):** Agent calls `book_flight`, gets a minimal response (just a booking ID, no confirmation details), and tells the user the flight is booked — without calling `verify_booking` to confirm it actually went through.
- **Verified booking (pass):** Agent calls `book_flight`, gets the same minimal response, then calls `verify_booking` to confirm the booking succeeded before responding to the user.
- **Self-confirming action (pass):** Agent calls `search_and_book`, which returns a comprehensive response including confirmation status, price, and flight details. Skipping `verify_booking` is reasonable here — the action tool's response already confirmed success.

In [ ]:
# --- Failing trace: agent books but doesn't verify ---
@mlflow.trace(name="travel_agent", span_type=SpanType.AGENT)
def verification_skipped_unverified(messages: list[dict]):
    root_span = mlflow.get_current_active_span()
    mlflow.tracing.set_span_chat_tools(root_span, TRAVEL_AGENT_TOOLS)
    mlflow.update_current_trace(
        tags={"failure_mode": "verification_skipped", "expected_result": "fail"}
    )

    with mlflow.start_span(name="search_flights", span_type=SpanType.TOOL) as span:
        span.set_inputs({"from_city": "NYC", "to_city": "London", "date": "2026-08-15"})
        span.set_outputs([{"flight_id": "FL-301", "airline": "BA", "price": 480}])

    with mlflow.start_span(name="book_flight", span_type=SpanType.TOOL) as span:
        span.set_inputs({"flight_id": "FL-301"})
        span.set_outputs({"booking_id": "BK-901"})

    return (
        "Your flight is booked! NYC to London on August 15, flight FL-301. "
        "Booking reference: BK-901."
    )


# --- Passing trace: agent books and verifies ---
@mlflow.trace(name="travel_agent", span_type=SpanType.AGENT)
def verification_skipped_verified(messages: list[dict]):
    root_span = mlflow.get_current_active_span()
    mlflow.tracing.set_span_chat_tools(root_span, TRAVEL_AGENT_TOOLS)
    mlflow.update_current_trace(
        tags={"failure_mode": "verification_skipped", "expected_result": "pass"}
    )

    with mlflow.start_span(name="search_flights", span_type=SpanType.TOOL) as span:
        span.set_inputs({"from_city": "NYC", "to_city": "London", "date": "2026-08-15"})
        span.set_outputs([{"flight_id": "FL-301", "airline": "BA", "price": 480}])

    with mlflow.start_span(name="book_flight", span_type=SpanType.TOOL) as span:
        span.set_inputs({"flight_id": "FL-301"})
        span.set_outputs({"booking_id": "BK-902"})

    with mlflow.start_span(name="verify_booking", span_type=SpanType.TOOL) as span:
        span.set_inputs({"booking_id": "BK-902"})
        span.set_outputs({
            "booking_id": "BK-902",
            "status": "confirmed",
            "flight_id": "FL-301",
        })

    return (
        "Your flight is booked and confirmed! NYC to London on August 15, "
        "flight FL-301. Booking BK-902 is confirmed."
    )


# --- Passing trace: action tool already confirms, verification not needed ---
@mlflow.trace(name="travel_agent", span_type=SpanType.AGENT)
def verification_skipped_self_confirming(messages: list[dict]):
    root_span = mlflow.get_current_active_span()
    mlflow.tracing.set_span_chat_tools(root_span, TRAVEL_AGENT_TOOLS)
    mlflow.update_current_trace(
        tags={"failure_mode": "verification_skipped", "expected_result": "pass"}
    )

    with mlflow.start_span(name="search_and_book", span_type=SpanType.TOOL) as span:
        span.set_inputs({"from_city": "NYC", "to_city": "London", "date": "2026-08-15"})
        span.set_outputs({
            "booking_id": "BK-903",
            "flight_id": "FL-302",
            "airline": "BA",
            "price": 450,
            "departure": "08:00",
            "arrival": "20:00",
            "status": "confirmed",
        })

    return (
        "Your flight is booked and confirmed! NYC to London on August 15, "
        "flight FL-302, 08:00-20:00. Booking BK-903 is confirmed."
    )


verification_skipped_unverified([
    {"role": "user", "content": "Book me a flight from NYC to London on August 15."}
])
verification_skipped_verified([
    {"role": "user", "content": "Book me a flight from NYC to London on August 15."}
])
verification_skipped_self_confirming([
    {"role": "user", "content": "Book me a flight from NYC to London on August 15."}
])
mlflow.flush_trace_async_logging()
print("Created 3 traces (1 fail, 2 pass)")

### Load traces

We fetch the Verification Skipped traces — one failing (unverified booking) and two passing (verified booking + self-confirming action).

In [ ]:
verification_traces = mlflow.search_traces(
    experiment_ids=[EXPERIMENT.experiment_id],
    filter_string="tags.failure_mode = 'verification_skipped'",
    return_type="list",
)

print(f"Traces found: {len(verification_traces)}")
for t in verification_traces:
    tags = t.info.tags or {}
    print(
        f"  [{tags.get('expected_result', '?')}] Input: {str(t.info.request_preview)[:80]}"
    )
    print(f"    Output: {str(t.info.response_preview)[:100]}")
    print()

### Building the `verification_skipped` judge

No existing MLflow scorer checks whether an agent verified its actions. We build a custom LLM judge using `make_judge()` that reads the trace and assesses whether verification was warranted and performed.

The judge considers:
- Did the agent take a state-changing action (booking, cancellation, etc.)?
- Was a verification tool available in the agent's toolset?
- Did the action tool's response already provide sufficient confirmation?
- If not, did the agent call the verification tool?

In [ ]:
from typing import Literal

from mlflow.genai.judges import make_judge

verification_skipped_judge = make_judge(
    name="verification_skipped",
    instructions=(
        "You are evaluating whether an AI agent properly verified the result of its actions.\n\n"
        "The agent is a travel booking assistant. Its available tools are described in the trace. "
        "One of the available tools is a verification tool (e.g., `verify_booking`) that can confirm "
        "whether a previous action succeeded.\n\n"
        "Evaluate the agent's behavior:\n"
        "1. Identify whether the agent took a state-changing action (e.g., booking a flight, "
        "cancelling a booking).\n"
        "2. Check whether a verification tool was available in the agent's toolset.\n"
        "3. Examine the action tool's response. If the response already includes comprehensive "
        "confirmation details (e.g., status: confirmed, full flight details, price), then "
        "separate verification is not necessary — the action tool already confirmed success.\n"
        "4. If the action tool's response was minimal (e.g., just a booking ID with no confirmation "
        "status), the agent should have called the verification tool to confirm the action succeeded "
        "before reporting success to the user.\n\n"
        "User's request: {{ inputs }}\n"
        "Agent's response: {{ outputs }}\n"
        "Agent's trace (including available tools and tool calls): {{ trace }}\n\n"
        "Return 'yes' if the agent either verified its action or verification was not needed "
        "(because the action tool already confirmed success). "
        "Return 'no' if the agent skipped verification when it was warranted."
    ),
    model="openai:/gpt-4o",
    feedback_value_type=Literal["yes", "no"],
)

with mlflow.start_run(run_name="verification-skipped-judge") as run:
    results = mlflow.genai.evaluate(
        data=verification_traces,
        scorers=[verification_skipped_judge],
    )

print_eval_results(results, "verification_skipped", EXPERIMENT.experiment_id)

### Interpreting the results

- **Unverified booking** → `no` — the agent called `book_flight` and got a minimal response (just a booking ID), but never called `verify_booking` to confirm the booking succeeded. It told the user the flight was booked without knowing for sure.
- **Verified booking** → `yes` — the agent called `book_flight`, got the same minimal response, then called `verify_booking` which confirmed the booking. The agent verified before reporting success.
- **Self-confirming action** → `yes` — the agent called `search_and_book` which returned comprehensive details including `"status": "confirmed"`. Separate verification was not needed — the action tool already confirmed success.

**The nuance:** Not every action requires a separate verification step. The judge reasons about whether the action tool's response was sufficient. A minimal response (just a booking ID) warrants verification; a comprehensive confirmed response does not. This is why a deterministic check ("was `verify_booking` called?") would be too rigid — it would flag the self-confirming case as a failure when it's actually fine.

**Why metrics is empty:** `make_judge()` with `feedback_value_type=Literal["yes", "no"]` sets `aggregations=[]` — no aggregation is defined for categorical string values. If you need a numeric mean, use `feedback_value_type=bool` instead. The per-trace verdicts and rationales are the primary output for this scorer.

**Note:** This scorer uses an LLM judge, so results may vary slightly between runs. The verdicts above are the expected outcomes for these traces, but LLM judges are non-deterministic — borderline cases may occasionally be judged differently.

For full details on this failure mode and how the judge works, see [verification_skipped.md](verification_skipped.md).